In [2]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))

Saving test_NP.txt to test_NP.txt
Saving train_NP.txt to train_NP.txt
Uploaded files: ['test_NP.txt', 'train_NP.txt']


In [3]:
#Read and parse the dataset
def load_dataset(filename):
    sentences = []
    tags = []

    current_words = []
    current_tags = []

    with open(filename, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            # Blank line indicates the end of a sentence
            if not line:
                if current_words:
                    sentences.append(current_words)
                    tags.append(current_tags)

                    current_words = []
                    current_tags = []
                continue

            parts = line.split()

            if len(parts) != 2:
                continue

            word, tag = parts
            current_words.append(word)
            current_tags.append(tag)

    # Add the final sentence if the file has no trailing blank line
    if current_words:
        sentences.append(current_words)
        tags.append(current_tags)

    return sentences, tags


train_sentences, train_tags = load_dataset("train_NP.txt")
test_sentences, test_tags = load_dataset("test_NP.txt")

print("Training sentences:", len(train_sentences))
print("Testing sentences:", len(test_sentences))

print("\nFirst training sentence:")
print(train_sentences[0])
print(train_tags[0])

print("\nFirst testing sentence:")
print(test_sentences[0])
print(test_tags[0])

Training sentences: 823
Testing sentences: 77

First training sentence:
['Rockwell', 'International', 'Corp.', "'s", 'Tulsa', 'unit', 'said', 'it', 'signed', 'a', 'tentative', 'agreement', 'extending', 'its', 'contract', 'with', 'Boeing', 'Co.', 'to', 'provide', 'structural', 'parts', 'for', 'Boeing', "'s", '747', 'jetliners', '.']
['B-NP', 'I-NP', 'I-NP', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'B-NP', 'I-NP', 'I-NP', 'O']

First testing sentence:
['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to', 'take', 'another', 'sharp', 'dive', 'if', 'trade', 'figures', 'for', 'September', ',', 'due', 'for', 'release', 'tomorrow', ',', 'fail', 'to', 'show', 'a', 'substantial', 'improvement', 'from', 'July', 'and', 'August', "'s", 'near-record', 'deficits', '.']
['B-NP', 'O', 'B-NP', 'I-NP', 'O', 'O', 'O', 'O', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'B-NP',

In [4]:
#Check the dataset
from collections import Counter

all_tags = sorted({
    tag
    for sentence_tags in train_tags
    for tag in sentence_tags
})

print("Available tags:", all_tags)

print("\nTraining tag distribution:")
print(Counter(tag for seq in train_tags for tag in seq))

print("\nTesting tag distribution:")
print(Counter(tag for seq in test_tags for tag in seq))

assert len(train_sentences) == len(train_tags)
assert len(test_sentences) == len(test_tags)

for words, labels in zip(train_sentences, train_tags):
    assert len(words) == len(labels)

for words, labels in zip(test_sentences, test_tags):
    assert len(words) == len(labels)

print("\nDataset checks passed.")

Available tags: ['B-NP', 'I-NP', 'O']

Training tag distribution:
Counter({'O': 8394, 'I-NP': 5727, 'B-NP': 5051})

Testing tag distribution:
Counter({'O': 842, 'I-NP': 579, 'B-NP': 475})

Dataset checks passed.


Train the HMM from scratch

In [5]:
#Count initial states, transitions and emissions
from collections import Counter, defaultdict
import numpy as np
import math

# Add-K smoothing parameter
K = 1.0

# Hidden states
states = sorted({
    tag
    for sentence_tags in train_tags
    for tag in sentence_tags
})

num_states = len(states)
state_to_idx = {tag: i for i, tag in enumerate(states)}

# Vocabulary from training data
vocab = {
    word
    for sentence in train_sentences
    for word in sentence
}

# Unknown-word symbol
UNK = "<UNK>"
vocab.add(UNK)
vocab_size = len(vocab)

# Count initial tags, transitions, emissions and tag frequencies
initial_counts = Counter()
transition_counts = defaultdict(Counter)
emission_counts = defaultdict(Counter)
tag_counts = Counter()

for words, labels in zip(train_sentences, train_tags):

    if not labels:
        continue

    # Initial tag count
    initial_counts[labels[0]] += 1

    for i, (word, tag) in enumerate(zip(words, labels)):
        tag_counts[tag] += 1

        # Emission count: word given tag
        emission_counts[tag][word] += 1

        # Transition count: current tag given previous tag
        if i > 0:
            previous_tag = labels[i - 1]
            transition_counts[previous_tag][tag] += 1

num_train_sentences = len(train_sentences)

print("Number of states:", num_states)
print("Vocabulary size:", vocab_size)
print("Initial counts:", dict(initial_counts))
print("Tag counts:", dict(tag_counts))

Number of states: 3
Vocabulary size: 4571
Initial counts: {'B-NP': 541, 'O': 282}
Tag counts: {'B-NP': 5051, 'I-NP': 5727, 'O': 8394}


In [6]:
#Calculate the HMM probabilities

# Initial probabilities
initial_probs = {}

for tag in states:
    initial_probs[tag] = (
        initial_counts[tag] + K
    ) / (
        num_train_sentences + K * num_states
    )


# Transition probabilities
transition_probs = {}

for prev_tag in states:
    transition_probs[prev_tag] = {}

    for curr_tag in states:
        transition_probs[prev_tag][curr_tag] = (
            transition_counts[prev_tag][curr_tag] + K
        ) / (
            tag_counts[prev_tag] + K * num_states
        )


# Emission probabilities
emission_probs = {}

for tag in states:
    emission_probs[tag] = {}

    for word in vocab:
        emission_probs[tag][word] = (
            emission_counts[tag][word] + K
        ) / (
            tag_counts[tag] + K * vocab_size
        )


print("Initial probabilities:")
for tag, prob in initial_probs.items():
    print(f"{tag}: {prob:.6f}")

print("\nTransition probability matrix:")
print("Rows = previous tag; columns = current tag")

print("        ", states)
for prev_tag in states:
    print(
        prev_tag,
        [
            round(transition_probs[prev_tag][curr_tag], 5)
            for curr_tag in states
        ]
    )

print("\nSample emission probabilities:")
for tag in states:
    print(tag, {
        word: round(emission_probs[tag][word], 6)
        for word in list(vocab)[:5]
    })

Initial probabilities:
B-NP: 0.656174
I-NP: 0.001211
O: 0.342615

Transition probability matrix:
Rows = previous tag; columns = current tag
         ['B-NP', 'I-NP', 'O']
B-NP [0.02968, 0.69529, 0.27483]
I-NP [0.0459, 0.38656, 0.5651]
O [0.48827, 0.00012, 0.41539]

Sample emission probabilities:
B-NP {'repurchased': 0.000104, 'banned': 0.000104, 'Oil': 0.000208, 'draw': 0.000104, 'vice': 0.000624}
I-NP {'repurchased': 9.7e-05, 'banned': 9.7e-05, 'Oil': 9.7e-05, 'draw': 9.7e-05, 'vice': 0.000971}
O {'repurchased': 0.000154, 'banned': 0.000231, 'Oil': 7.7e-05, 'draw': 0.000154, 'vice': 7.7e-05}


In [7]:
#Implement the Forward algorithm
def logsumexp(values):
    """Stable calculation of log(sum(exp(values)))."""
    values = np.asarray(values, dtype=float)
    maximum = np.max(values)

    if np.isneginf(maximum):
        return -np.inf

    return maximum + np.log(np.sum(np.exp(values - maximum)))


def get_log_emission(word, tag):
    """Return log P(word | tag), including unseen words."""
    if word not in vocab:
        word = UNK

    return math.log(emission_probs[tag][word])


def forward_algorithm(words):
    """
    Calculate the log probability and probability
    of a sentence using the Forward algorithm.
    """
    if not words:
        return 0.0, 1.0

    T = len(words)
    N = len(states)

    # alpha[t, j] = log probability of observations
    # up to position t, ending in state j
    alpha = np.full((T, N), -np.inf)

    # Initialization
    for j, tag in enumerate(states):
        alpha[0, j] = (
            math.log(initial_probs[tag])
            + get_log_emission(words[0], tag)
        )

    # Recursion
    for t in range(1, T):
        for j, curr_tag in enumerate(states):

            previous_values = []

            for i, prev_tag in enumerate(states):
                value = (
                    alpha[t - 1, i]
                    + math.log(
                        transition_probs[prev_tag][curr_tag]
                    )
                )
                previous_values.append(value)

            alpha[t, j] = (
                logsumexp(previous_values)
                + get_log_emission(words[t], curr_tag)
            )

    # Termination
    log_probability = logsumexp(alpha[T - 1, :])

    # May underflow to 0 for extremely long sentences
    probability = (
        math.exp(log_probability)
        if log_probability > -745
        else 0.0
    )

    return log_probability, probability

In [8]:
#Calculate probabilities for every test sentence
forward_results = []

for sentence_number, words in enumerate(test_sentences, start=1):
    log_prob, prob = forward_algorithm(words)

    forward_results.append({
        "sentence_number": sentence_number,
        "sentence": " ".join(words),
        "log_probability": log_prob,
        "probability": prob
    })

print("Forward algorithm results for the first 10 sentences:\n")

for result in forward_results[:10]:
    print("Sentence:", result["sentence"])
    print("Log probability:", result["log_probability"])
    print("Probability:", result["probability"])
    print("-" * 60)

print("Total test sentences:", len(forward_results))

Forward algorithm results for the first 10 sentences:

Sentence: Confidence in the pound is widely expected to take another sharp dive if trade figures for September , due for release tomorrow , fail to show a substantial improvement from July and August 's near-record deficits .
Log probability: -263.67713340609185
Probability: 3.0653209800359964e-115
------------------------------------------------------------
Sentence: Chancellor of the Exchequer Nigel Lawson 's restated commitment to a firm monetary policy has helped to prevent a freefall in sterling over the past week .
Log probability: -188.57167926654162
Probability: 1.2716284954963693e-82
------------------------------------------------------------
Sentence: But analysts reckon underlying support for sterling has been eroded by the chancellor 's failure to announce any new policy measures in his Mansion House speech last Thursday .
Log probability: -213.48865720067374
Probability: 1.918908332992834e-93
-------------------------

In [9]:
#Implement Viterbi decoding
def viterbi_decode(words):
    """Predict the most likely tag sequence for a sentence."""

    if not words:
        return []

    T = len(words)
    N = len(states)

    # delta[t, j] stores the best log score ending in state j
    delta = np.full((T, N), -np.inf)

    # Store the previous state selected for each position
    backpointer = np.zeros((T, N), dtype=int)

    # Initialization
    for j, tag in enumerate(states):
        delta[0, j] = (
            math.log(initial_probs[tag])
            + get_log_emission(words[0], tag)
        )

    # Recursion
    for t in range(1, T):
        for j, curr_tag in enumerate(states):

            candidate_scores = []

            for i, prev_tag in enumerate(states):
                score = (
                    delta[t - 1, i]
                    + math.log(
                        transition_probs[prev_tag][curr_tag]
                    )
                    + get_log_emission(words[t], curr_tag)
                )

                candidate_scores.append(score)

            best_previous_state = int(np.argmax(candidate_scores))

            delta[t, j] = candidate_scores[best_previous_state]
            backpointer[t, j] = best_previous_state

    # Termination: select the best final state
    best_last_state = int(np.argmax(delta[T - 1, :]))

    # Backtracking
    best_path = [best_last_state]

    for t in range(T - 1, 0, -1):
        best_last_state = backpointer[t, best_last_state]
        best_path.append(best_last_state)

    best_path.reverse()

    return [states[index] for index in best_path]

In [10]:
#Predict tags for the test dataset
predicted_tags = [
    viterbi_decode(sentence)
    for sentence in test_sentences
]

print("Original sentence:")
print(" ".join(test_sentences[0]))

print("\nActual tags:")
print(test_tags[0])

print("\nPredicted tags:")
print(predicted_tags[0])

assert len(predicted_tags) == len(test_tags)

for words, predictions in zip(test_sentences, predicted_tags):
    assert len(words) == len(predictions)

print("\nAll test sentences have been tagged.")

Original sentence:
Confidence in the pound is widely expected to take another sharp dive if trade figures for September , due for release tomorrow , fail to show a substantial improvement from July and August 's near-record deficits .

Actual tags:
['B-NP', 'O', 'B-NP', 'I-NP', 'O', 'O', 'O', 'O', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'O', 'O', 'O', 'B-NP', 'B-NP', 'O', 'O', 'O', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'B-NP', 'I-NP', 'I-NP', 'O']

Predicted tags:
['B-NP', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'O', 'O', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'O', 'O', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'O', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'O']

All test sentences have been tagged.


Evaluate the HMM model

In [11]:
#Calculate precision, recall and F1-score
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support
)

# Flatten the actual and predicted tags
actual_flat = [
    tag for sentence in test_tags for tag in sentence
]

predicted_flat = [
    tag for sentence in predicted_tags for tag in sentence
]

# Token-level accuracy
accuracy = accuracy_score(actual_flat, predicted_flat)

# Metrics for each tag
precision, recall, f1, support = precision_recall_fscore_support(
    actual_flat,
    predicted_flat,
    labels=["B-NP", "I-NP", "O"],
    zero_division=0
)

print("HMM Evaluation Results")
print("=" * 45)
print(f"Token Accuracy: {accuracy:.4f}")

for i, tag in enumerate(["B-NP", "I-NP", "O"]):
    print(f"\nTag: {tag}")
    print(f"Precision: {precision[i]:.4f}")
    print(f"Recall:    {recall[i]:.4f}")
    print(f"F1-score:  {f1[i]:.4f}")
    print(f"Support:   {support[i]}")

print("\nClassification Report:")
print(classification_report(
    actual_flat,
    predicted_flat,
    labels=["B-NP", "I-NP", "O"],
    zero_division=0
))

HMM Evaluation Results
Token Accuracy: 0.8497

Tag: B-NP
Precision: 0.7728
Recall:    0.8737
F1-score:  0.8202
Support:   475

Tag: I-NP
Precision: 0.8516
Recall:    0.8325
F1-score:  0.8419
Support:   579

Tag: O
Precision: 0.9004
Recall:    0.8480
F1-score:  0.8734
Support:   842

Classification Report:
              precision    recall  f1-score   support

        B-NP       0.77      0.87      0.82       475
        I-NP       0.85      0.83      0.84       579
           O       0.90      0.85      0.87       842

    accuracy                           0.85      1896
   macro avg       0.84      0.85      0.85      1896
weighted avg       0.85      0.85      0.85      1896



In [12]:
#Save predictions to a file
output_file = "NP_predictions.txt"

with open(output_file, "w", encoding="utf-8") as file:
    for words, actual, predicted in zip(
        test_sentences, test_tags, predicted_tags
    ):
        for word, actual_tag, predicted_tag in zip(
            words, actual, predicted
        ):
            file.write(
                f"{word}\t{actual_tag}\t{predicted_tag}\n"
            )

        file.write("\n")

print("Predictions saved to:", output_file)

from google.colab import files
files.download(output_file)

Predictions saved to: NP_predictions.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>